# 03 — Establish baselines, train models, and select on validation data

This notebook compares every candidate on the same untouched chronological
validation period. **MAE** is the primary selection metric. RMSE emphasizes large
misses; R² compares with a constant mean; QLIKE is a scale-free volatility loss;
bias reveals systematic over- or underprediction.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError("Run this notebook from the project root or notebooks folder.")

# This fallback makes the src-layout package importable even before an editable install.
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
import matplotlib.pyplot as plt

from market_volatility.data import REPORTS_DIR, chronological_split, load_modeling_dataset
from market_volatility.evaluate import forecast_metrics, prediction_frame
from market_volatility.features import FORWARD_HORIZON, TARGET_COLUMN
from market_volatility.plotting import save_figure
from market_volatility.train import (
    baseline_predictions,
    fit_models,
    model_specs,
    predict_models,
)

VALIDATION_START = "2016-01-01"
TEST_START = "2021-01-01"

## Models and baselines

Baselines:

1. **Training mean:** no time variation.
2. **Persistence:** next-20-day volatility equals current trailing 20-day RV.
3. **Historical volatility:** next-20-day volatility equals trailing 60-day RV.

Candidates:

1. **HAR-style linear model:** future RV from 5/20/60-day RV. Transparent and
   grounded in volatility persistence across horizons.
2. **Ridge:** a scaled linear model using every feature; shrinkage limits noisy
   coefficients.
3. **Random forest:** nonlinear interactions and thresholds, with regularized
   leaves.
4. **Histogram gradient boosting:** a second nonlinear tree method that learns
   residual structure sequentially.

The settings are deliberately modest. A serious tuning exercise would use
walk-forward cross-validation inside the training era—not the test period.

In [ ]:
modeling_data = load_modeling_dataset()
splits = chronological_split(
    modeling_data,
    validation_start=VALIDATION_START,
    test_start=TEST_START,
    purge_horizon=FORWARD_HORIZON,
)

fitted_models = fit_models(splits.train)

validation_predictions = baseline_predictions(
    splits.validation,
    training_target_mean=float(splits.train[TARGET_COLUMN].mean()),
).join(predict_models(fitted_models, splits.validation))

validation_metrics = forecast_metrics(
    splits.validation[TARGET_COLUMN], validation_predictions
)
display(validation_metrics.style.format("{:.4f}"))

In [ ]:
candidate_names = list(model_specs())
selected_model = validation_metrics.loc[candidate_names, "MAE"].idxmin()
print(f"Selected ML candidate by validation MAE: {selected_model}")

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
validation_metrics.to_csv(REPORTS_DIR / "validation_metrics.csv")
validation_table = prediction_frame(
    splits.validation[TARGET_COLUMN], validation_predictions
)
validation_table.to_csv(
    REPORTS_DIR / "validation_predictions.csv", index_label="Date"
)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
validation_metrics["MAE"].sort_values().plot.bar(ax=ax, color="steelblue")
ax.set(title="Validation MAE (lower is better)", ylabel="MAE", xlabel="")
ax.grid(axis="y", alpha=0.25)
save_figure(fig, "03_validation_mae")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(
    splits.validation.index,
    splits.validation[TARGET_COLUMN],
    label="Realized next-20-day volatility",
    linewidth=1.2,
    color="black",
)
ax.plot(
    validation_predictions.index,
    validation_predictions[selected_model],
    label=selected_model,
    linewidth=1.0,
)
ax.plot(
    validation_predictions.index,
    validation_predictions["baseline_persistence_rv20"],
    label="persistence baseline",
    linewidth=0.9,
    alpha=0.8,
)
ax.set(title="Validation forecasts", ylabel="Annualized volatility", xlabel="Date")
ax.legend()
ax.grid(alpha=0.25)
save_figure(fig, "03_validation_forecasts")
plt.show()

The selected name is now fixed. Notebook 04 may open the test period once,
refit that model using all permissible pre-test observations, and report the
final result. Do not return to this notebook to choose another model after seeing
test performance; that would turn the test set into another validation set.